This notebook is a mostly Spikeinterface pipeline to analyze data from Dan Denman's lab.

Data from two probes in a four-probe experiment are provided. This notebook analyzes ProbeA, which has a bit more drift and is therefore a more interesting example. 

Check the instructions [**here**](https://spikeinterface.readthedocs.io/en/0.93.0/overview.html).

Be sure to checkout the other [tutorials](https://spikeinterface.readthedocs.io/en/0.93.0/getting_started/plot_getting_started.html#) and the [how to guide](https://spikeinterface.readthedocs.io/en/latest/how_to/analyze_neuropixels.html). 


In [ ]:


import spikeinterface.full as si
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

job_kwargs = dict(n_jobs=-1, chunk_duration='1s', progress_bar=True) # how to chunk and process data

# specify the of the record node folder 
base_folder = Path(r'Z:\Dan_OEP_example\D9_example')

# output_parent
output_parent = Path(r'Z:\Dan_OEP_example\SI_output')
motion_output = output_parent.joinpath('motion_data')
if not os.path.exists(output_parent):
    os.mkdir(output_parent)
if not os.path.exists(motion_output):
    print('making motion output folder')
    os.mkdir(motion_output)

oep_folder = base_folder
# load oep data
stream_names, stream_ids = si.get_neo_streams('openephysbinary', oep_folder)
probeA_name = [s for s in stream_names if "ProbeA" in s]

if len(probeA_name) == 1:
    raw_rec = si.read_openephys(oep_folder, stream_name=probeA_name[0], load_sync_channel=False)
else:
    print("No stream for ProbeA")
rec_preprocess = si.bandpass_filter(recording=raw_rec, freq_min=300, freq_max=9000)
rec_preprocess = si.phase_shift(recording=rec_preprocess)
rec_preprocess = si.common_reference(recording=rec_preprocess)

rec_corrected = si.correct_motion(recording=rec_preprocess, preset="rigid_fast", folder=motion_output, overwrite = True, **job_kwargs)

Load motion data from the motion output folder for plotting

In [ ]:
peaks = np.load(motion_output.joinpath('peaks.npy'))  # labeled tuple for each peak, including sample index and amplitude
peak_loc = np.load(motion_output.joinpath('peak_locations.npy')) # labeled tupple for each peak, x and y coordinates in um
disp_seg0 = np.load(motion_output.joinpath('motion').joinpath('displacement_seg0.npy')) # displacement of the whole probe (rigid => only one spatial segment)
time_bins_sec = np.load(motion_output.joinpath('motion').joinpath('temporal_bins_s_seg0.npy'))

In [ ]:
# Plot the displacement, to check for whether the inferred motion looks reasonable
plt.plot(time_bins_sec, disp_seg0)
plt.xlabel('time (sec)')
plt.ylabel('displacement (um)')

### Is the inferred motion 'reasonable'?
Most of time, movement of the probe relative to the tissue will be slow (5-10 um over 60-100 sec), and won't recover instantly. Sharp deflections (>20 um) in the inferred motion are likely to be due to problems with the measurement. 
We can check whether the motion looks 'real' by looking at the extracted spikes. SI outputs that info in the peaks.npy and peak_location.npy files loaded above. New units 'turning on' can look like a motion event.

In [ ]:
# make drift/raster plot of spike y (vertical) lcoations vs. time
x = peak_loc['x']
y = peak_loc['y']
t = peaks['sample_index'].astype(float)/30000 + time_bins_sec[0]  # convert peaks times to seconds, add the start time (this is approximate)
amp = np.abs(peaks['amplitude'])*0.195           # amplitudes are given in bits, and are the negative deflections. convert to uV
color_min = np.percentile(amp,10)
color_max = np.percentile(amp,90)

plt.scatter(t,y,c=amp, vmin=color_min, vmax=color_max, s=1)   # s sets te size of the symbols
plt.xlabel('time (sec)')    # these are sample numbers in this segment
plt.ylabel('vertical position (um)')
plt.colorbar()

In this example, the change at 7400 sec does not look like consistent motion across the probe, and the change ~7600 looks like some units turning on in the upper layers--but it could also be that the probe has moved closer to those units. Overall, This was a rigid sorting model. We'll try sorting with KS4 motion correction and 2 'blocks' to allow adjustments for the top and bottom half of the probe shanks.

In [ ]:
ks4_params = si.get_default_sorter_params('kilosort4')
ks4_params['do_CAR'] = False # skip CAR (performed above)
ks4_params['nblocks'] = 2    # run KS4 motion correction with two segments (top and bottom of the probe)
for k in ks4_params.keys():
    print(f'   - {k}: {ks4_params[k]}')

Sort the preprocessed (but not motion corrected) by specifying rec_preprocess.
Keep KS4 preprocessing to get the correct whitening and scaling.

In [ ]:
# sort the preprocessed data. Set rec_sort to rec_corrected to sort the motion corrected data
# 
rec_sort = rec_preprocess
#rec_sort = rec_sort.save(folder=output_parent / 'preprocess', format='binary', **job_kwargs)
sort_folder = output_parent / 'kilosort4_output'
# run ks4
sorting = si.run_sorter('kilosort4', rec_sort, folder=sort_folder,
                        docker_image=False, verbose=True, **ks4_params)
sorting = si.read_sorter_folder(sort_folder)

analyzer = si.create_sorting_analyzer(sorting, rec_sort, sparse=True, format="memory")
# compute waveforms 
analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=500)
analyzer.compute("waveforms",  ms_before=1.5,ms_after=1.5, **job_kwargs)
analyzer.compute("templates", operators=["average", "median", "std"])
analyzer.compute("noise_levels")
analyzer.compute("correlograms")
analyzer.compute("unit_locations")
analyzer.compute("spike_amplitudes", **job_kwargs)
metric_names=['firing_rate', 'presence_ratio', 'snr', 'isi_violation', 'amplitude_cutoff']
metrics = si.compute_quality_metrics(analyzer, metric_names=metric_names)
# save
analyzer_saved = analyzer.save_as(folder=output_parent / "analyzer", format="binary_folder")



Next, create file of spike times in seconds to build PSTHs from the stimulus data in D9_example/grating_df.csv

In [ ]:
# path to the kilosort ouput:
probeA_folder = probeA_name[0].split(sep='#')[1]   #split out the folder name from the stream name
print(probeA_folder)
phy_output = sort_folder.joinpath('sorter_output')
spike_times_samples = np.load(phy_output.joinpath('spike_times.npy'))
# path to timestamps for this probe. These names are hardcoded except for stream name
ts_folder = base_folder.joinpath('Record Node 101').joinpath('experiment1').joinpath('recording3').joinpath('continuous').joinpath(probeA_folder)
timestamps_sec = np.load(ts_folder.joinpath('timestamps.npy'))
num_samples = timestamps_sec.size
# spikes can be assigned times that are > number of timestamps. Set these to the max 
spikes_past_max = spike_times_samples > (num_samples-1)
print(f'{np.sum(spikes_past_max)} spikes have times > max timestamp')
spike_times_samples[spikes_past_max] = num_samples-1
spike_times_sec = timestamps_sec[spike_times_samples]

np.save(phy_output.joinpath('spike_times_sec.npy'),spike_times_sec)